In [1]:
import numpy as np
from alc import diagRH
from alc import traspuesta
from alc import norma
from alc import inversa
from alc import producto_interno

In [2]:
def obtenerQ2d(imgPts,objPts):
  matriz_Q=[] #Armo mi matriz Q en donde voy a guardar las filas creadas

  for i in range(len(objPts)): #Separo las coordenadas y aplico el algoritmo
    U= objPts[i][0]
    V= objPts[i][1]
    u= imgPts[i][0]
    v= imgPts[i][1]

    fila_1=[U, V, 1, 0, 0, 0, -u*U, -u*V, -u]
    fila_2=[0, 0, 0, U, V, 1, -v*U, -v*V, -v]

    matriz_Q.append(fila_1)
    matriz_Q.append(fila_2)

  return np.array(matriz_Q)

In [3]:
def encontrar_vector_minimo(array):
    #Armo mi matriz Q*T.Q=A
    Q_final=traspuesta(array) @ array
    #Llamo a la funcion diagRH  para quedarme con el menor autovalor
    S,D =diagRH(Q_final)
    
    menor= D[0,0]
    indice=0
    for i in range(1,D.shape[0]):#Busco el menor autovalor en la diagonal 
        if D[i][i] < menor:
            menor=  D[i,i]
            indice=i
    vector_h= S[:,indice]
    return vector_h #No lo transformo asi puedo reutilizar esta funcion

In [4]:
def armarA(Hlst):
    filas_A = []

    for H in Hlst:
        # Primera columna de la homografía
        a, b, c = H[:, 0]  #selecciona todas  las filas agarra la primer columna
        # Segunda columna de la homografía
        d, e, f = H[:, 1] #selecciona todas las filas agarra la segunda columna

        #c1*TxBxc2
        fila_1 = [ 
            a*d,
            a*e + b*d,
            b*e,
            a*f + c*d,
            b*f + c*e,
            c*f
        ]
        #c1*TxBxc1-c2*TxBxc2=0
        #Ver demo despues pero se re entendio   
        fila_2 = [
            a*a - d*d,
            2*(a*b - d*e),
            b*b - e*e,
            2*(a*c - d*f), 
            2*(b*c - e*f),
            c*c - f*f
        ]
        filas_A.append(fila_1)
        filas_A.append(fila_2)
    return np.array(filas_A)

In [5]:
def normalizar_diagonal(B, autovalores, tol=1e-12):
    lista_positivo = []
    lista_negativo = []
    lista_cero = []

    for valor in autovalores:
        if valor > tol:
            lista_positivo.append(valor)
        elif valor < -tol:
            lista_negativo.append(valor)
        else:
            lista_cero.append(valor)

    if len(lista_cero) > 0:
        # raise lanza una excepción(un problema  ocurrido en la ejecucion)
        # valueerror tipo de exepcion
        raise ValueError("B tiene autovalores cercanos a cero")

    elif len(lista_positivo) > 0 and len(lista_negativo) > 0:
        raise ValueError("B es indefinida: no se puede aplicar Cholesky")

    elif len(lista_negativo) > 0:
        return -B

    else:
        return B

In [6]:
def calcularK2D(A):
    b_final= encontrar_vector_minimo(A)
    B = np.array([
    [b_final[0], b_final[1], b_final[3]],
    [b_final[1], b_final[2], b_final[4]],
    [b_final[3], b_final[4], b_final[5]]])
    #Necesito aplicar cholesky para ello se que B  
    _, D =diagRH(B)

    lista_autovalores=[]
    for i in range(D.shape[0]):
        lista_autovalores.append(D[i][i])
    B=normalizar_diagonal(B,lista_autovalores)

    #Aca tengo que aplicar el algoritmo de cholesky (luego lo creo)
    L = np.linalg.cholesky(B)
    K  = traspuesta(inversa(L))
    if K[2][2]!=1:
        K=K/K[2][2]
    return K


In [7]:
def obtenerPose2D(H, K):
    K_inv = inversa(K)
    c1 = H[:, 0]
    c2 = H[:, 1]
    c3 = H[:, 2]

    # Multiplicar por la inversa de K
    v1 = K_inv @ c1
    v2 = K_inv @ c2
    v3 = K_inv @ c3

    # Calcular el factor de escala
    if norma(v1) < 1e-12:
        raise ValueError("No se puede calcular lambda")
    
    lamda = 1 / norma(v1)
    if (lamda * v3)[2] < 0: #Chequeo que el valor de lamda sea positivo para  que  la camara apunta adelante
        lamda = -lamda
                
    # Recuperar r1, r2 y t
    r1 = lamda * v1
    r2 = lamda * v2
    t = lamda * v3

    r1 = r1 / norma(r1) #Normalizo ||r1||
    r2 = r2 - producto_interno(r1, r2) * r1   

    if norma(r2) < 1e-12:
        raise ValueError("No se puede construir la rotación")

    r2 = r2 / norma(r2) #Normalizo ||r2||

    #Armo ||r3|| como r1Xr2 ya que tiene que ser  ortogonal
    r3 = np.array([ 
        r1[1]*r2[2] - r1[2]*r2[1],
        r1[2]*r2[0] - r1[0]*r2[2],
        r1[0]*r2[1] - r1[1]*r2[0]])

    #Contruyo la matriz de rotación
    R = np.zeros((3, 3))

    for i in range(3):
        R[i, 0] = r1[i]
        R[i, 1] = r2[i]
        R[i, 2] = r3[i]

    return R, t

In [12]:
import pandas as pd

# read_csv ya usa la primera fila (pts;X;Y;Z;u_0;v_0;...;u_3;v_3) como nombres de las columnas,
# asi que no hay que saltearla: con skiprows=1 se perdia el encabezado y el punto 6 pasaba a ser el encabezado
data = pd.read_csv('/home/luis/PycharmProjects/Modulo_ALC/TP/salguero/world_img_pts.csv', delimiter=';')

# Cada fila de data es un punto y cada columna una coordenada. data.iloc[i] devuelve la fila i (el punto i),
# por eso tomamos las columnas por nombre. np.array las pasa de columna de pandas a array de numpy

# Coordenadas del mundo real de cada punto, en metros
X = np.array(data['X'])
Y = np.array(data['Y'])
Z = np.array(data['Z'])

# Coordenadas de cada punto en la imagen de cada vista, en pixeles
U0 = np.array(data['u_0'])
V0 = np.array(data['v_0'])
U1 = np.array(data['u_1'])
V1 = np.array(data['v_1'])
U2 = np.array(data['u_2'])
V2 = np.array(data['v_2'])
U3 = np.array(data['u_3'])
V3 = np.array(data['v_3'])


obtpts = np.column_stack((X,Y))
imgPts0 = np.column_stack((U0,V0))
imgPts1 = np.column_stack((U1,V1))
imgPts2 = np.column_stack((U2,V2))
imgPts3 = np.column_stack((U3,V3))

print("X =", X)
print("Y =", Y)
print("Z =", Z)
print("vista 0: u =", U0, " v =", V0)
print("vista 1: u =", U1, " v =", V1)
print("vista 2: u =", U2, " v =", V2)
print("vista 3: u =", U3, " v =", V3)

X = [-4.   4.   6.5 -5.  -5. ]
Y = [ 3  0  3 14 76]
Z = [0.  0.  0.1 0.1 0.1]
vista 0: u = [ 281  889 1139  418  833]  v = [474 521 493 385 262]
vista 1: u = [ 255 1071 1403  437  997]  v = [490 577 550 377 227]
vista 2: u = [ 469 1147 1425  627 1094]  v = [581 653 625 485 357]
vista 3: u = [ 446 1118 1396  600 1064]  v = [563 621 590 466 331]


In [13]:
Q0 = obtenerQ2d(obtpts,imgPts0)
Q1 = obtenerQ2d(obtpts,imgPts1)
Q2 = obtenerQ2d(obtpts,imgPts2)
Q3 = obtenerQ2d(obtpts,imgPts3)


In [15]:
vmin0 = encontrar_vector_minimo(Q0)
vmin1 = encontrar_vector_minimo(Q1)
vmin2 = encontrar_vector_minimo(Q2)
vmin3 = encontrar_vector_minimo(Q3)

Hmin0 = vmin0.reshape(3, 3)
Hmin1 = vmin1.reshape(3, 3)
Hmin2 = vmin2.reshape(3, 3)
Hmin3 = vmin3.reshape(3, 3)


In [18]:
A = armarA([Hmin0, Hmin1, Hmin2, Hmin3])
A

array([[ 1.07952508e-07, -4.71384696e-07, -6.18875097e-08,
         1.69453567e-08,  1.01728341e-08, -2.99924326e-10],
       [-9.95123099e-08,  1.53327138e-06, -3.40463512e-06,
        -6.05723833e-08,  2.61964128e-07, -5.03051289e-09],
       [ 9.20658713e-08, -3.84859244e-07, -1.50674526e-07,
         1.33169717e-08,  1.62562549e-08, -3.97461050e-10],
       [-9.56618958e-08,  1.41686684e-06, -3.11799539e-06,
        -5.43876147e-08,  2.33040537e-07, -4.34646448e-09],
       [ 7.92447554e-08, -3.08462132e-07, -1.01263366e-07,
         1.20412201e-08,  1.04013078e-08, -2.55598132e-10],
       [-8.34244126e-08,  1.13096551e-06, -2.30775971e-06,
        -4.66545621e-08,  1.87676876e-07, -3.81365363e-09],
       [ 8.09055487e-08, -3.33490338e-07, -6.26290127e-08,
         1.25801160e-08,  7.46626999e-09, -1.97150038e-10],
       [-8.05819074e-08,  1.14232288e-06, -2.41668946e-06,
        -4.58244498e-08,  1.90690618e-07, -3.75901503e-09]])

In [19]:
K = calcularK2D(A)
K

array([[  3.8128754 ,  -0.81631362,   5.63793517],
       [  0.        ,   3.61023953, -25.29665396],
       [  0.        ,   0.        ,   1.        ]])

In [20]:
datos = np.loadtxt('/home/luis/PycharmProjects/Modulo_ALC/TP/salguero/gs_10_09_point_correspondences_v5.txt')  # (6, 51)
puntos = datos.T #TODO: usar la funciont raspuesta               # (51, 6): una fila por punto
objPts = puntos[:, 1:4]       # (51, 3) X Y Z  ->  puntos[:, 1:3] para obtenerQ2d
imgPts = puntos[:, 4:6]       # (51, 2) u v

In [24]:
Q_ejercicio_b = obtenerQ2d(objPts,imgPts)
vmin_ejercicio_b = encontrar_vector_minimo(Q_ejercicio_b)
Hminejercicio_b = vmin_ejercicio_b.reshape(3, 3)
Hminejercicio_b

array([[-3.02421918e-04, -5.53751163e-04,  4.37705579e-01],
       [-4.37014267e-05,  1.80450202e-03, -8.98980598e-01],
       [-3.71351923e-07, -8.15297456e-05,  1.56220003e-02]])

In [25]:
pose = obtenerPose2D(Hminejercicio_b,K)
pose

(array([[-0.98425284,  0.11198268,  0.13677071],
        [-0.17671004, -0.60380131, -0.77730145],
        [-0.00446196, -0.78922992,  0.61408162]]),
 array([  742.80872954, -1676.71549531,   187.70554143]))